# DS5 Step 3 — Corrected Personalization / Intervention-Fitting Analysis

**Stage:** Stage 4, Step 3  
**Corrected primary moderator:** pre-existing NetC `cluster_net_only`  

**What "NetC" is:** the full-sample NeT-only 2-cluster "robustness check for Phase 1 split" that notebook `101_ds5_network_subphenotype.ipynb` (Path C) computed and saved as `cluster_net_only` in `ds5_net_subcluster_assignments.csv` (see that notebook's cell 32-37 / Figure C7). This is a genuine, pre-existing, outcome-free artifact -- **not** a new computation -- but it is a *different* clustering from Path C's own more widely-cited within-profile B1/B2 (Profile B) / A1/A2 (Profile A) sub-clustering result. Renamed from the ambiguous bare "Path C" (used in an earlier draft of this notebook) to "NetC" here specifically to avoid that collision, parallel to "Net2" (the strict two-feature sensitivity computed fresh below).
**Primary outcome:** `x_mean`

## Approved corrected scope

1. `cluster_net_only × study_condition → x_mean` — **primary**.
2. `cluster_net_only × study_condition → commitment_score` — replication.
3. `cluster_net_only × study_condition → chi_behav` — tertiary.
4. A newly constructed phenotype using exactly `HHI_net` + `rho_self` — pre-specified sensitivity on `x_mean` only.

The earlier `profile_p1` and `profile_k3` results are retained as provenance, but explicitly classified as exploratory/non-confirmatory because their moderators were constructed using behavioral variables that overlap the tested outcomes.

Framework C remains deferred after the ABM validation failure. Framework D remains unimplemented. Nothing in `src/personalization/` is changed.

## Prospective multiplicity hierarchy

The corrected inferential family contains the **three distinct outcome interactions under the pre-existing NetC moderator**: `x_mean`, `commitment_score`, and `chi_behav`. Holm adjustment is applied across these three tests. This family is prospectively frozen for the corrected run, but Step 3 remains exploratory rather than preregistered confirmation.

The strict two-feature analysis is a robustness check of the already-tested primary `x_mean` estimand under an alternate network-only labeling, not a fourth outcome discovery hypothesis. Its p-value is reported unadjusted and cannot independently establish a new claim; it is interpreted only for agreement with the corrected primary.

The four legacy interactions retain their original four-test Holm adjustment solely to reproduce their historical output. They are excluded from the corrected family and are not claim-bearing.

In [ ]:
# 1. Imports, paths, and frozen constants
import datetime
import warnings
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import rankdata
from sklearn.cluster import KMeans
from sklearn.metrics import (
    adjusted_rand_score, silhouette_score,
    calinski_harabasz_score, davies_bouldin_score,
)
from sklearn.preprocessing import StandardScaler
from IPython.display import display, Markdown

warnings.filterwarnings('ignore', category=FutureWarning)

ROOT = Path('..').resolve()
MODEL_ROOT = ROOT / 'outputs' / 'model_outputs'
PHENO_DIR = MODEL_ROOT / 'phenotyping'
OUTPUT_DIR = MODEL_ROOT / 'personalization'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONDITIONS = ['control', 'default', 'reintervention']
ALPHA = 0.05
RANDOM_STATE = 42
N_BOOTSTRAPS = 2000
N_STABILITY_BOOTSTRAPS = 100
SAMPLE_FRAC = 0.80
EXPECTED_N = 414
MIN_CLUSTER_SIZE = 40
MIN_INTERACTION_CELL = 30
JACCARD_STABLE = 0.75

NET2_FEATURES = ['HHI_net', 'rho_self']
LEGACY_PHASE1_FEATURES = ['sig_hat', 'H_emp', 'x_mean', 'HHI_net', 'rho_self']

print('Repository root:', ROOT)
print('Corrected output directory:', OUTPUT_DIR)

## 1. Moderator construction and validation — outcomes intentionally unavailable

This section loads only participant IDs, condition, and network features. No `x_mean`, `sig_hat`, `commitment_score`, or `chi_behav` column is loaded while either network moderator is validated, constructed, or named.

- NetC labels are pre-existing and were fit on all N=463 NeT-eligible participants from `HHI_net`, `rho_self`, `H_trans`, and `H_app`.
- The strict sensitivity uses N=414 and exactly `HHI_net` plus `rho_self`.
- Standardization, K-means initialization, k-range diagnostics, and selection thresholds match Phase 1 conventions.
- Labels are ordered only by network centroids.

In [ ]:
# 2. Load network-only inputs using explicit columns (no behavioral outcome access)
feature_path = PHENO_DIR / 'ds5_phenotype_feature_matrix.csv'
path_c_path = PHENO_DIR / 'ds5_net_subcluster_assignments.csv'
path_c_stability_path = PHENO_DIR / 'ds5_net_subcluster_stability.csv'

for path in [feature_path, path_c_path, path_c_stability_path]:
    assert path.exists(), f'Missing input: {path}'

network_base = pd.read_csv(
    feature_path,
    usecols=['participant_id_canonical', 'study_condition'] + NET2_FEATURES,
)
path_c = pd.read_csv(
    path_c_path,
    usecols=['participant_id_canonical', 'study_condition', 'cluster_net_only'],
)
path_c_stability = pd.read_csv(path_c_stability_path)

assert len(network_base) == EXPECTED_N
assert network_base['participant_id_canonical'].is_unique
assert network_base[NET2_FEATURES].notna().all().all()
assert set(network_base.columns) == {
    'participant_id_canonical', 'study_condition', 'HHI_net', 'rho_self'
}
assert len(path_c) == EXPECTED_N and path_c['participant_id_canonical'].is_unique
assert path_c['cluster_net_only'].notna().all()

network_base = network_base.merge(
    path_c.rename(columns={'study_condition': 'study_condition_path_c'}),
    on='participant_id_canonical', how='left', validate='one_to_one',
)
assert (network_base['study_condition'] == network_base['study_condition_path_c']).all()
network_base = network_base.drop(columns='study_condition_path_c')

for forbidden in ['x_mean', 'sig_hat', 'commitment_score', 'chi_behav']:
    assert forbidden not in network_base.columns

print('Network-only construction table:', network_base.shape)
print('Columns available before moderator freeze:', network_base.columns.tolist())

In [ ]:
# 3. Give the pre-existing NetC numeric partition semantic names using network centroids only
path_c_centroids = (
    network_base.groupby('cluster_net_only')[NET2_FEATURES]
    .mean().sort_values(['HHI_net', 'rho_self'])
)
assert len(path_c_centroids) == 2
path_c_name_map = {
    path_c_centroids.index[0]: 'NetC Distributed',
    path_c_centroids.index[1]: 'NetC Hub-locked',
}
network_base['profile_path_c'] = network_base['cluster_net_only'].map(path_c_name_map)

path_c_full_stability = path_c_stability.loc[
    path_c_stability['population'] == 'Full NeT-eligible'
].iloc[0]

print('NetC centroids used for naming:')
display(path_c_centroids)
print(
    f"Pre-existing full-sample stability: Jaccard="
    f"{path_c_full_stability.jaccard_mean:.4f} ± "
    f"{path_c_full_stability.jaccard_std:.4f}"
)

In [ ]:
# 4. Strict two-feature construction: same Phase 1 scaling/clustering conventions
strict_scaler = StandardScaler()
X_net2 = strict_scaler.fit_transform(network_base[NET2_FEATURES].to_numpy())

strict_metric_rows = []
strict_labels_by_k = {}
for k in range(2, 7):
    model = KMeans(
        n_clusters=k, init='k-means++', n_init=10,
        random_state=RANDOM_STATE,
    )
    labels = model.fit_predict(X_net2)
    sizes = np.bincount(labels)
    strict_labels_by_k[k] = labels
    strict_metric_rows.append({
        'k': k,
        'silhouette': float(silhouette_score(X_net2, labels)),
        'calinski_harabasz': float(calinski_harabasz_score(X_net2, labels)),
        'davies_bouldin': float(davies_bouldin_score(X_net2, labels)),
        'min_cluster_size': int(sizes.min()),
        'cluster_sizes': '|'.join(map(str, sizes.tolist())),
        'eligible_phase1_rule': bool(
            silhouette_score(X_net2, labels) > 0.20 and sizes.min() >= MIN_CLUSTER_SIZE
        ),
    })

strict_diagnostics = pd.DataFrame(strict_metric_rows)
eligible = strict_diagnostics[strict_diagnostics['eligible_phase1_rule']]
assert len(eligible) > 0
SELECTED_NET2_K = int(eligible.loc[eligible['silhouette'].idxmax(), 'k'])
assert SELECTED_NET2_K == 2, 'Strict sensitivity expected a two-profile solution'

strict_raw_labels = strict_labels_by_k[SELECTED_NET2_K]
strict_centroids = (
    network_base.assign(_raw_cluster=strict_raw_labels)
    .groupby('_raw_cluster')[NET2_FEATURES]
    .mean().sort_values(['HHI_net', 'rho_self'])
)
strict_raw_to_profile = {
    strict_centroids.index[0]: 'Net2 Distributed',
    strict_centroids.index[1]: 'Net2 Hub-locked',
}
strict_raw_to_cluster = {
    strict_centroids.index[0]: 0,
    strict_centroids.index[1]: 1,
}
network_base['cluster_net2'] = [strict_raw_to_cluster[x] for x in strict_raw_labels]
network_base['profile_net2'] = [strict_raw_to_profile[x] for x in strict_raw_labels]

print('Strict k diagnostics (outcomes still unavailable):')
display(strict_diagnostics.round(4))
print('Selected network centroids:')
display(strict_centroids)

In [ ]:
# 5. Phase 1-style co-membership stability for the frozen strict partition
def co_membership_jaccard_stability(X, full_labels, k, n_boot=100, sample_frac=0.80, seed=42):
    rng = np.random.RandomState(seed)
    n = len(X)
    scores = []
    for b in range(n_boot):
        idx = rng.choice(n, int(n * sample_frac), replace=False)
        boot_model = KMeans(
            n_clusters=k, init='k-means++', n_init=5, random_state=b
        )
        boot_labels = boot_model.fit_predict(X[idx])
        reference = full_labels[idx]
        tri_i, tri_j = np.triu_indices(len(idx), k=1)
        same_reference = reference[tri_i] == reference[tri_j]
        same_boot = boot_labels[tri_i] == boot_labels[tri_j]
        denominator = np.sum(same_reference | same_boot)
        scores.append(
            np.sum(same_reference & same_boot) / denominator if denominator else 0.0
        )
    return float(np.mean(scores)), float(np.std(scores))


strict_jaccard_mean, strict_jaccard_std = co_membership_jaccard_stability(
    X_net2, strict_raw_labels, SELECTED_NET2_K,
    n_boot=N_STABILITY_BOOTSTRAPS,
    sample_frac=SAMPLE_FRAC,
    seed=RANDOM_STATE,
)

strict_diagnostics['selected'] = strict_diagnostics['k'] == SELECTED_NET2_K
strict_diagnostics['selected_jaccard_mean'] = np.where(
    strict_diagnostics['selected'], strict_jaccard_mean, np.nan
)
strict_diagnostics['selected_jaccard_std'] = np.where(
    strict_diagnostics['selected'], strict_jaccard_std, np.nan
)

print(
    f'Strict Net2 stability: Jaccard={strict_jaccard_mean:.4f} ± '
    f'{strict_jaccard_std:.4f}'
)

In [ ]:
# 6. Condition independence and cell-size gates before any outcome is loaded
def cramers_v(table, chi2):
    n = table.to_numpy().sum()
    denominator = n * min(table.shape[0] - 1, table.shape[1] - 1)
    return float(np.sqrt(chi2 / denominator)) if denominator > 0 else np.nan


def moderator_condition_audit(frame, moderator):
    table = pd.crosstab(frame[moderator], frame['study_condition']).reindex(
        columns=CONDITIONS, fill_value=0
    )
    chi2, p_value, dof, expected = stats.chi2_contingency(table)
    return {
        'moderator': moderator,
        'table': table,
        'chi2': float(chi2),
        'df': int(dof),
        'p_value': float(p_value),
        'cramers_v': cramers_v(table, chi2),
        'min_observed': int(table.to_numpy().min()),
        'min_expected': float(expected.min()),
    }


path_c_audit = moderator_condition_audit(network_base, 'profile_path_c')
strict_audit = moderator_condition_audit(network_base, 'profile_net2')
network_audits = {
    'profile_path_c': path_c_audit,
    'profile_net2': strict_audit,
}

path_c_gate_pass = (
    float(path_c_full_stability.jaccard_mean) >= JACCARD_STABLE and
    path_c_audit['p_value'] >= ALPHA and
    path_c_audit['min_observed'] >= MIN_INTERACTION_CELL
)
strict_gate_pass = (
    strict_jaccard_mean >= JACCARD_STABLE and
    strict_audit['p_value'] >= ALPHA and
    strict_audit['min_observed'] >= MIN_INTERACTION_CELL
)

for name, audit in network_audits.items():
    print(f'\n{name} × study_condition')
    display(audit['table'])
    print(
        f"chi2({audit['df']})={audit['chi2']:.4f}, "
        f"p={audit['p_value']:.6f}, Cramer's V={audit['cramers_v']:.4f}, "
        f"min observed={audit['min_observed']}"
    )

strict_vs_path_c_ari = adjusted_rand_score(
    network_base['profile_net2'], network_base['profile_path_c']
)
print(f'\nARI(strict Net2, pre-existing NetC)={strict_vs_path_c_ari:.6f}')
print('NetC eligibility gate:', path_c_gate_pass)
print('Strict Net2 eligibility gate:', strict_gate_pass)

assert path_c_gate_pass
assert strict_gate_pass

In [ ]:
# 7. Export the frozen strict phenotype and diagnostics before outcomes are introduced
strict_assignments = network_base[[
    'participant_id_canonical', 'study_condition',
    'cluster_net2', 'profile_net2', 'HHI_net', 'rho_self',
]].copy()

strict_diagnostics['condition_chi2'] = np.where(
    strict_diagnostics['selected'], strict_audit['chi2'], np.nan
)
strict_diagnostics['condition_df'] = np.where(
    strict_diagnostics['selected'], strict_audit['df'], np.nan
)
strict_diagnostics['condition_p'] = np.where(
    strict_diagnostics['selected'], strict_audit['p_value'], np.nan
)
strict_diagnostics['condition_cramers_v'] = np.where(
    strict_diagnostics['selected'], strict_audit['cramers_v'], np.nan
)
strict_diagnostics['min_condition_cell'] = np.where(
    strict_diagnostics['selected'], strict_audit['min_observed'], np.nan
)
strict_diagnostics['all_preoutcome_gates_pass'] = np.where(
    strict_diagnostics['selected'], strict_gate_pass, False
)

strict_assignments.to_csv(
    OUTPUT_DIR / 'strict_net2_assignments.csv', index=False
)
strict_diagnostics.to_csv(
    OUTPUT_DIR / 'strict_net2_cluster_diagnostics.csv', index=False,
    float_format='%.10g',
)

print('Frozen and exported strict network phenotype before outcome loading.')

## 2. Outcome loading — only after both moderator definitions pass their gates

The moderator labels above are now frozen. Behavioral outcomes and legacy phenotype assignments are loaded only from this point onward.

In [ ]:
# 8. Load outcomes and legacy provenance assignments after moderator freeze
source_specs = {
    'stop': (
        MODEL_ROOT / 'stochastic_processes' / 'participant_parameters.csv',
        ['participant_id_canonical', 'study_condition', 'x_mean'],
    ),
    'stm': (
        MODEL_ROOT / 'statistical_mechanics' / 'participant_stm_parameters.csv',
        ['participant_id_canonical', 'study_condition', 'x_mean', 'chi_behav'],
    ),
    'gat': (
        MODEL_ROOT / 'game_theory' / 'participant_gat_parameters.csv',
        ['participant_id_canonical', 'study_condition', 'commitment_score', 'strategy'],
    ),
    'p1': (
        PHENO_DIR / 'ds5_cluster_assignments.csv',
        ['participant_id_canonical', 'study_condition', 'profile_p1'],
    ),
    'k3': (
        PHENO_DIR / 'ds5_gmm_k3_assignments.csv',
        ['participant_id_canonical', 'study_condition', 'profile_k3'],
    ),
    'path_a': (
        PHENO_DIR / 'ds5_behav_only_cluster_assignments.csv',
        ['participant_id_canonical', 'study_condition', 'cluster_behav'],
    ),
    'path_f': (
        PHENO_DIR / 'ds5_hierarchical_assignments.csv',
        ['participant_id_canonical', 'study_condition', 'cell_f'],
    ),
    'path_e': (
        PHENO_DIR / 'ds5_condition_cluster_assignments.csv',
        ['participant_id_canonical', 'study_condition', 'cond_cluster_full'],
    ),
}

loaded = {}
for name, (path, columns) in source_specs.items():
    assert path.exists(), f'Missing required source: {path}'
    loaded[name] = pd.read_csv(path, usecols=columns)
    assert loaded[name]['participant_id_canonical'].is_unique

analysis = network_base.copy()
for name, frame in loaded.items():
    right = frame.copy().rename(columns={'study_condition': f'study_condition__{name}'})
    if name == 'stm':
        right = right.rename(columns={'x_mean': 'x_mean__stm_check'})
    analysis = analysis.merge(
        right, on='participant_id_canonical', how='left', validate='one_to_one'
    )
    condition_col = f'study_condition__{name}'
    assert analysis[condition_col].notna().all()
    assert (analysis[condition_col] == analysis['study_condition']).all()

analysis = analysis.drop(columns=[c for c in analysis if c.startswith('study_condition__')])

assert len(analysis) == EXPECTED_N
assert analysis['participant_id_canonical'].is_unique
assert analysis[[
    'x_mean', 'commitment_score', 'chi_behav', 'profile_p1', 'profile_k3',
    'cluster_behav', 'cell_f', 'cond_cluster_full', 'strategy',
]].notna().all().all()
assert np.allclose(analysis['x_mean'], analysis['x_mean__stm_check'], atol=1e-12)

print('Outcome/legacy analysis table:', analysis.shape)
print(f'Correlation(x_mean, commitment_score)={analysis.x_mean.corr(analysis.commitment_score):.4f}')

# Disclosure: how much does the corrected moderator actually differ from the legacy one?
_netc_vs_p1 = adjusted_rand_score(analysis['profile_path_c'], analysis['profile_p1'])
print('\nCrosstab NetC (corrected) vs profile_p1 (legacy):')
print(pd.crosstab(analysis['profile_path_c'], analysis['profile_p1']))
print(f'ARI(NetC, profile_p1) = {_netc_vs_p1:.4f}')


## 3. Interaction methods

The original statistical guardrails are retained:

- cellwise Shapiro–Wilk plus median-centered Levene as the parametric gate;
- aligned rank transform (ART) interaction fallback when the gate fails;
- raw OLS and ART estimates both exported;
- partial η² and a 2,000-iteration participant-stratified bootstrap CI;
- corrected three-outcome Holm family;
- hierarchical pairwise condition follow-ups only after a family-adjusted omnibus result;
- sensitivity and legacy rows excluded from new claim generation.

In [ ]:
# 9. Statistical helpers
def categorical_design(moderator, condition, include_interaction=True):
    a = pd.Categorical(moderator)
    b = pd.Categorical(condition)
    a_dummy = pd.get_dummies(a, drop_first=True, dtype=float).to_numpy()
    b_dummy = pd.get_dummies(b, drop_first=True, dtype=float).to_numpy()
    blocks = [np.ones((len(a), 1)), a_dummy, b_dummy]
    if include_interaction:
        interactions = np.column_stack([
            a_dummy[:, i] * b_dummy[:, j]
            for i in range(a_dummy.shape[1])
            for j in range(b_dummy.shape[1])
        ])
        blocks.append(interactions)
    return np.column_stack(blocks)


def nested_interaction_test(frame, outcome, moderator, use_art=False):
    data = frame[[outcome, moderator, 'study_condition']].dropna().copy()
    y_raw = data[outcome].to_numpy(dtype=float)
    x_reduced = categorical_design(data[moderator], data['study_condition'], False)
    x_full = categorical_design(data[moderator], data['study_condition'], True)
    y = y_raw.copy()
    if use_art:
        beta_additive = np.linalg.lstsq(x_reduced, y_raw, rcond=None)[0]
        y = rankdata(y_raw - x_reduced @ beta_additive, method='average')
    beta_reduced = np.linalg.lstsq(x_reduced, y, rcond=None)[0]
    beta_full = np.linalg.lstsq(x_full, y, rcond=None)[0]
    sse_reduced = float(np.sum((y - x_reduced @ beta_reduced) ** 2))
    sse_full = float(np.sum((y - x_full @ beta_full) ** 2))
    rank_reduced = int(np.linalg.matrix_rank(x_reduced))
    rank_full = int(np.linalg.matrix_rank(x_full))
    df_effect = rank_full - rank_reduced
    df_error = len(data) - rank_full
    ss_interaction = max(0.0, sse_reduced - sse_full)
    f_stat = (ss_interaction / df_effect) / (sse_full / df_error)
    return {
        'F': float(f_stat),
        'p': float(stats.f.sf(f_stat, df_effect, df_error)),
        'partial_eta2': float(ss_interaction / (ss_interaction + sse_full)),
        'df_effect': int(df_effect),
        'df_error': int(df_error),
        'n': int(len(data)),
    }


def assumption_gate(frame, outcome, moderator):
    groups = []
    shapiro_p = {}
    for (level, condition), group in frame.groupby(
        [moderator, 'study_condition'], observed=True, sort=True
    ):
        values = group[outcome].dropna().to_numpy(dtype=float)
        p_value = float(stats.shapiro(values).pvalue)
        groups.append(values)
        shapiro_p[f'{level}|{condition}'] = p_value
    levene = stats.levene(*groups, center='median')
    minimum = min(shapiro_p.values())
    passed = minimum >= ALPHA and float(levene.pvalue) >= ALPHA
    return {
        'assumptions_pass': bool(passed),
        'min_shapiro_p': float(minimum),
        'n_shapiro_below_alpha': int(sum(p < ALPHA for p in shapiro_p.values())),
        'levene_stat': float(levene.statistic),
        'levene_p': float(levene.pvalue),
    }


def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    sorted_p = p_values[order]
    adjusted_sorted = np.maximum.accumulate(
        (len(p_values) - np.arange(len(p_values))) * sorted_p
    )
    adjusted_sorted = np.minimum(adjusted_sorted, 1.0)
    adjusted = np.empty(len(p_values), dtype=float)
    adjusted[order] = adjusted_sorted
    return adjusted


def bootstrap_eta2(frame, outcome, moderator, use_art, rng, n_bootstraps):
    data = frame[[outcome, moderator, 'study_condition']].dropna().reset_index(drop=True)
    strata = [g.index.to_numpy() for _, g in data.groupby(
        [moderator, 'study_condition'], observed=True, sort=True
    )]
    estimates = []
    for _ in range(n_bootstraps):
        sampled = np.concatenate([
            rng.choice(indices, size=len(indices), replace=True) for indices in strata
        ])
        sample = data.loc[sampled].reset_index(drop=True)
        estimates.append(nested_interaction_test(
            sample, outcome, moderator, use_art=use_art
        )['partial_eta2'])
    low, high = np.percentile(estimates, [2.5, 97.5])
    return float(low), float(high)


def eta_label(value):
    if value >= 0.14:
        return 'large'
    if value >= 0.06:
        return 'medium'
    if value >= 0.01:
        return 'small'
    return 'below conventional small threshold'


def condition_audit_for_any(frame, moderator):
    return moderator_condition_audit(frame, moderator)


print('Statistical helpers defined.')

In [ ]:
# 10. Freeze corrected, sensitivity, and legacy analysis specifications
corrected_specs = [
    {
        'analysis_id': 'corrected_primary_path_c_x_mean',
        'result_tier': 'corrected_inferential_family',
        'analysis_role': 'primary',
        'moderator': 'profile_path_c',
        'outcome': 'x_mean',
        'feature_overlap_flag': False,
        'feature_overlap_reason': 'NetC uses network/app-structure features only',
        'claim_eligible': True,
    },
    {
        'analysis_id': 'corrected_replication_path_c_commitment_score',
        'result_tier': 'corrected_inferential_family',
        'analysis_role': 'replication',
        'moderator': 'profile_path_c',
        'outcome': 'commitment_score',
        'feature_overlap_flag': False,
        'feature_overlap_reason': 'NetC uses network/app-structure features only',
        'claim_eligible': True,
    },
    {
        'analysis_id': 'corrected_tertiary_path_c_chi_behav',
        'result_tier': 'corrected_inferential_family',
        'analysis_role': 'tertiary',
        'moderator': 'profile_path_c',
        'outcome': 'chi_behav',
        'feature_overlap_flag': False,
        'feature_overlap_reason': 'NetC uses network/app-structure features only',
        'claim_eligible': True,
    },
]

sensitivity_specs = [
    {
        'analysis_id': 'corrected_sensitivity_strict_net2_x_mean',
        'result_tier': 'corrected_sensitivity_non_discovery',
        'analysis_role': 'pre-specified sensitivity',
        'moderator': 'profile_net2',
        'outcome': 'x_mean',
        'feature_overlap_flag': False,
        'feature_overlap_reason': 'strict phenotype uses exactly HHI_net and rho_self',
        'claim_eligible': False,
    },
]

legacy_specs = [
    {
        'analysis_id': 'legacy_primary_profile_p1_x_mean',
        'result_tier': 'legacy_provenance_nonconfirmatory',
        'analysis_role': 'legacy original primary',
        'moderator': 'profile_p1', 'outcome': 'x_mean',
        'feature_overlap_flag': True,
        'feature_overlap_reason': 'x_mean entered five-feature profile_p1 construction and label naming',
        'claim_eligible': False,
    },
    {
        'analysis_id': 'legacy_replication_profile_p1_commitment_score',
        'result_tier': 'legacy_provenance_nonconfirmatory',
        'analysis_role': 'legacy original replication',
        'moderator': 'profile_p1', 'outcome': 'commitment_score',
        'feature_overlap_flag': True,
        'feature_overlap_reason': 'commitment_score correlates -0.9797 with construction input x_mean',
        'claim_eligible': False,
    },
    {
        'analysis_id': 'legacy_tertiary_profile_p1_chi_behav',
        'result_tier': 'legacy_provenance_nonconfirmatory',
        'analysis_role': 'legacy original tertiary',
        'moderator': 'profile_p1', 'outcome': 'chi_behav',
        'feature_overlap_flag': True,
        'feature_overlap_reason': 'chi_behav is deterministic in sig_hat and x_mean; both entered profile_p1',
        'claim_eligible': False,
    },
    {
        'analysis_id': 'legacy_sensitivity_profile_k3_x_mean',
        'result_tier': 'legacy_provenance_nonconfirmatory',
        'analysis_role': 'legacy original sensitivity',
        'moderator': 'profile_k3', 'outcome': 'x_mean',
        'feature_overlap_flag': True,
        'feature_overlap_reason': 'profile_k3 GMM uses the five Phase 1 features including x_mean',
        'claim_eligible': False,
    },
]

legacy_audits = {
    moderator: condition_audit_for_any(analysis, moderator)
    for moderator in ['profile_p1', 'profile_k3']
}
all_audits = {**network_audits, **legacy_audits}

print('Corrected family size:', len(corrected_specs))
print('Sensitivity tests:', len(sensitivity_specs))
print('Legacy provenance rows:', len(legacy_specs))

In [ ]:
# 11. Run all omnibus interactions; multiplicity is applied by tier afterward
def run_specifications(specifications, bootstrap_seed):
    rng = np.random.default_rng(bootstrap_seed)
    rows = []
    for spec in specifications:
        moderator = spec['moderator']
        outcome = spec['outcome']
        gate = assumption_gate(analysis, outcome, moderator)
        ols_result = nested_interaction_test(analysis, outcome, moderator, use_art=False)
        art_result = nested_interaction_test(analysis, outcome, moderator, use_art=True)
        selected_method = (
            'OLS raw-scale interaction' if gate['assumptions_pass']
            else 'aligned rank transform interaction'
        )
        selected = ols_result if gate['assumptions_pass'] else art_result
        boot_low, boot_high = bootstrap_eta2(
            analysis, outcome, moderator,
            use_art=not gate['assumptions_pass'],
            rng=rng, n_bootstraps=N_BOOTSTRAPS,
        )
        audit = all_audits[moderator]
        rows.append({
            **spec,
            'n': selected['n'],
            'n_moderator_levels': int(analysis[moderator].nunique()),
            'min_cell_n': audit['min_observed'],
            'moderator_condition_chi2': audit['chi2'],
            'moderator_condition_df': audit['df'],
            'moderator_condition_p': audit['p_value'],
            'moderator_condition_cramers_v': audit['cramers_v'],
            'assumptions_pass': gate['assumptions_pass'],
            'min_cell_shapiro_p': gate['min_shapiro_p'],
            'n_cells_shapiro_p_lt_0_05': gate['n_shapiro_below_alpha'],
            'levene_stat': gate['levene_stat'],
            'levene_p': gate['levene_p'],
            'selected_method': selected_method,
            'interaction_df': selected['df_effect'],
            'error_df': selected['df_error'],
            'interaction_F': selected['F'],
            'interaction_p_raw': selected['p'],
            'partial_eta2': selected['partial_eta2'],
            'partial_eta2_label': eta_label(selected['partial_eta2']),
            'partial_eta2_boot_ci_low': boot_low,
            'partial_eta2_boot_ci_high': boot_high,
            'bootstrap_iterations': N_BOOTSTRAPS,
            'ols_F': ols_result['F'],
            'ols_p': ols_result['p'],
            'ols_partial_eta2': ols_result['partial_eta2'],
            'art_F': art_result['F'],
            'art_p': art_result['p'],
            'art_partial_eta2': art_result['partial_eta2'],
        })
    return pd.DataFrame(rows)


corrected_results = run_specifications(corrected_specs, RANDOM_STATE + 10)
sensitivity_results = run_specifications(sensitivity_specs, RANDOM_STATE + 20)

# Preserve the previous bootstrap sequence and statistics for provenance rows.
legacy_results = run_specifications(legacy_specs, RANDOM_STATE)

corrected_results['interaction_p_family_adjusted'] = holm_adjust(
    corrected_results['interaction_p_raw']
)
corrected_results['multiplicity_family'] = 'corrected NetC three-outcome Holm family'
corrected_results['multiplicity_family_size'] = 3
corrected_results['family_adjustment_method'] = 'Holm'

sensitivity_results['interaction_p_family_adjusted'] = np.nan
sensitivity_results['multiplicity_family'] = 'none; robustness check of corrected primary estimand'
sensitivity_results['multiplicity_family_size'] = 0
sensitivity_results['family_adjustment_method'] = 'not applicable'

legacy_results['interaction_p_family_adjusted'] = holm_adjust(
    legacy_results['interaction_p_raw']
)
legacy_results['multiplicity_family'] = 'legacy original four-test Holm family; provenance only'
legacy_results['multiplicity_family_size'] = 4
legacy_results['family_adjustment_method'] = 'Holm (historical reproduction)'

interaction_results = pd.concat(
    [corrected_results, sensitivity_results, legacy_results],
    ignore_index=True,
)
interaction_results['significant_raw_0_05'] = interaction_results['interaction_p_raw'] < ALPHA
interaction_results['significant_family_adjusted_0_05'] = (
    interaction_results['interaction_p_family_adjusted'].notna() &
    (interaction_results['interaction_p_family_adjusted'] < ALPHA)
)

display(interaction_results[[
    'result_tier', 'analysis_role', 'moderator', 'outcome', 'n', 'min_cell_n',
    'selected_method', 'interaction_F', 'interaction_df', 'error_df',
    'interaction_p_raw', 'interaction_p_family_adjusted', 'partial_eta2',
    'partial_eta2_boot_ci_low', 'partial_eta2_boot_ci_high',
    'feature_overlap_flag', 'claim_eligible',
]].round(6))

In [ ]:
# 12. Hierarchical post-hoc interactions for Holm-significant family rows
posthoc_by_analysis = {}
for _, omnibus in interaction_results.iterrows():
    should_run = bool(omnibus['significant_family_adjusted_0_05'])
    if not should_run:
        posthoc_by_analysis[omnibus['analysis_id']] = []
        continue
    use_art = omnibus['selected_method'] == 'aligned rank transform interaction'
    pair_rows = []
    for condition_1, condition_2 in combinations(CONDITIONS, 2):
        subset = analysis[analysis['study_condition'].isin([condition_1, condition_2])]
        test = nested_interaction_test(
            subset, omnibus['outcome'], omnibus['moderator'], use_art=use_art
        )
        pair_rows.append({
            'condition_1': condition_1,
            'condition_2': condition_2,
            'F': test['F'],
            'df_effect': test['df_effect'],
            'df_error': test['df_error'],
            'p_raw': test['p'],
            'partial_eta2': test['partial_eta2'],
        })
    adjusted = holm_adjust([row['p_raw'] for row in pair_rows])
    for row, p_adjusted in zip(pair_rows, adjusted):
        row['p_holm_within_outcome'] = float(p_adjusted)
        row['significant_holm_0_05'] = bool(p_adjusted < ALPHA)
    posthoc_by_analysis[omnibus['analysis_id']] = pair_rows

interaction_results['posthoc_run'] = interaction_results['analysis_id'].map(
    lambda analysis_id: bool(posthoc_by_analysis[analysis_id])
)
interaction_results['posthoc_significant_condition_pairs'] = interaction_results['analysis_id'].map(
    lambda analysis_id: '; '.join(
        f"{row['condition_1']} vs {row['condition_2']} "
        f"(p_holm={row['p_holm_within_outcome']:.4g})"
        for row in posthoc_by_analysis[analysis_id]
        if row['significant_holm_0_05']
    )
)

for analysis_id, rows in posthoc_by_analysis.items():
    if rows:
        print(f'Post-hoc interaction contrasts: {analysis_id}')
        display(pd.DataFrame(rows).round(6))

## 4. Framework B descriptive tables

The original descriptive rows are retained unchanged in scope and relabeled as legacy where appropriate. Corrected NetC and strict Net2 cells are added. Condition-shaped Path A, Path F, Path E, and GaT strategy remain descriptive-only.

In [ ]:
# 13. Framework B cell summaries with separate RNG streams for legacy preservation
OUTCOMES = ['x_mean', 'commitment_score', 'chi_behav']

legacy_framework_specs = [
    ('profile_p1', 'Step 2 Phase 1', 'legacy provenance',
     'condition-independent but mixed-feature/outcome-leaking', True,
     'Non-confirmatory due to moderator-outcome leakage.'),
    ('profile_k3', 'Step 2 GMM v2 k=3', 'legacy provenance',
     'condition-independent but mixed-feature/outcome-leaking', True,
     'Non-confirmatory; G2 retained and underpowered.'),
    ('cluster_behav', 'Step 2 Path A', 'descriptive only',
     'condition-associated/post-treatment risk', True,
     'Do not interpret as trait moderation or a treatment rule.'),
    ('cell_f', 'Step 2 Path F', 'descriptive only',
     'condition-associated/post-treatment risk', True,
     'Do not interpret as trait moderation or a treatment rule.'),
    ('cond_cluster_full', 'Step 2 Path E', 'descriptive only',
     'defined within condition by construction', False,
     'Labels are not cross-condition comparable.'),
    ('strategy', 'Step 1 GaT', 'descriptive only',
     'strongly condition-shaped/post-treatment risk', True,
     'GaT strategy cannot support stable moderation or prescription.'),
]

corrected_framework_specs = [
    ('profile_path_c', 'Step 2 NetC full NeT-only', 'corrected primary companion',
     'condition-independent; pre-existing outcome-free construction', True,
     'Group-level corrected companion; network metrics remain study-period measurements.'),
    ('profile_net2', 'Step 3 strict HHI_net+rho_self sensitivity', 'corrected sensitivity companion',
     'condition-independent; pre-outcome gates passed', True,
     'Sensitivity only; cannot create a separate discovery claim.'),
]


def bootstrap_location_ci(values, rng, n_bootstraps):
    values = np.asarray(values, dtype=float)
    draws = rng.choice(values, size=(n_bootstraps, len(values)), replace=True)
    mean_low, mean_high = np.percentile(draws.mean(axis=1), [2.5, 97.5])
    median_low, median_high = np.percentile(np.median(draws, axis=1), [2.5, 97.5])
    return float(mean_low), float(mean_high), float(median_low), float(median_high)


def build_cell_rows(specifications, framework_tier, rng):
    rows = []
    for moderator, source, role, condition_status, comparable, guardrail in specifications:
        for outcome in OUTCOMES:
            for (condition, level), group in analysis.groupby(
                ['study_condition', moderator], observed=True, sort=True
            ):
                values = group[outcome].dropna().to_numpy(dtype=float)
                mean_low, mean_high, median_low, median_high = bootstrap_location_ci(
                    values, rng, N_BOOTSTRAPS
                )
                overlap = ''
                if moderator in ['profile_p1', 'profile_k3']:
                    if outcome == 'x_mean':
                        overlap = 'direct x_mean leakage into moderator construction'
                    elif outcome == 'commitment_score':
                        overlap = 'near-duplicate proxy of x_mean construction input'
                    elif outcome == 'chi_behav':
                        overlap = 'chi_behav is deterministic in sig_hat and x_mean construction inputs'
                rows.append({
                    'framework': 'B',
                    'framework_tier': framework_tier,
                    'moderator': moderator,
                    'moderator_source': source,
                    'analysis_role': role,
                    'condition_independence_status': condition_status,
                    'cross_condition_comparable': comparable,
                    'outcome': outcome,
                    'study_condition': condition,
                    'phenotype_level': level,
                    'n': int(len(values)),
                    'mean': float(np.mean(values)),
                    'mean_boot_ci_low': mean_low,
                    'mean_boot_ci_high': mean_high,
                    'median': float(np.median(values)),
                    'median_boot_ci_low': median_low,
                    'median_boot_ci_high': median_high,
                    'std': float(np.std(values, ddof=1)),
                    'q1': float(np.percentile(values, 25)),
                    'q3': float(np.percentile(values, 75)),
                    'bootstrap_iterations': N_BOOTSTRAPS,
                    'feature_overlap_warning': overlap,
                    'interpretation_guardrail': guardrail,
                })
    return rows


# Same seed/order as the original notebook for the 168 legacy/descriptive rows.
legacy_cell_rows = build_cell_rows(
    legacy_framework_specs, 'legacy_or_descriptive',
    np.random.default_rng(RANDOM_STATE + 1),
)
corrected_cell_rows = build_cell_rows(
    corrected_framework_specs, 'corrected',
    np.random.default_rng(RANDOM_STATE + 2),
)
cell_summary = pd.DataFrame(legacy_cell_rows + corrected_cell_rows)

assert len(legacy_cell_rows) == 168
assert len(corrected_cell_rows) == 36
assert len(cell_summary) == 204

for moderator in [x[0] for x in legacy_framework_specs + corrected_framework_specs]:
    for outcome in OUTCOMES:
        total = cell_summary.loc[
            (cell_summary['moderator'] == moderator) &
            (cell_summary['outcome'] == outcome), 'n'
        ].sum()
        assert int(total) == EXPECTED_N

print(f'Framework B rows: {len(cell_summary)} (168 retained + 36 corrected)')
display(cell_summary[
    (cell_summary['moderator'] == 'profile_path_c')
][[
    'outcome', 'study_condition', 'phenotype_level', 'n', 'mean',
    'mean_boot_ci_low', 'mean_boot_ci_high', 'median',
    'median_boot_ci_low', 'median_boot_ci_high',
]].round(4))

In [ ]:
# 14. Markdown formatting and result handles
def fmt_p(value):
    if pd.isna(value):
        return 'not applicable'
    if value < 0.0001:
        return f'{value:.2e}'
    return f'{value:.4f}'


def markdown_table(frame, columns=None, digits=4):
    table = frame.copy()
    if columns is not None:
        table = table[columns]
    for column in table.columns:
        if pd.api.types.is_float_dtype(table[column]):
            table[column] = table[column].map(
                lambda value: '' if pd.isna(value) else f'{value:.{digits}f}'
            )
    lines = [
        '| ' + ' | '.join(map(str, table.columns)) + ' |',
        '| ' + ' | '.join(['---'] * len(table.columns)) + ' |',
    ]
    for _, row in table.iterrows():
        lines.append('| ' + ' | '.join(str(row[c]) for c in table.columns) + ' |')
    return '\n'.join(lines)


def get_result(analysis_id):
    return interaction_results.loc[
        interaction_results['analysis_id'] == analysis_id
    ].iloc[0]


primary = get_result('corrected_primary_path_c_x_mean')
replication = get_result('corrected_replication_path_c_commitment_score')
tertiary = get_result('corrected_tertiary_path_c_chi_behav')
sensitivity = get_result('corrected_sensitivity_strict_net2_x_mean')

corrected_chi_posthoc = posthoc_by_analysis[tertiary.analysis_id]
corrected_chi_pair_text = '; '.join(
    f"{row['condition_1']} vs {row['condition_2']} "
    f"(Holm p={fmt_p(row['p_holm_within_outcome'])})"
    for row in corrected_chi_posthoc if row['significant_holm_0_05']
) or 'none'

print('Result handles ready.')

## 5. Publication figures — corrected moderator only

The figures below use only the pre-existing NetC network phenotype and its strict two-feature sensitivity. Legacy `profile_p1` / `profile_k3` results are intentionally excluded from claim-bearing graphics because of moderator–outcome feature leakage.

Each figure is saved as a 400-dpi PNG and a vector PDF under `outputs/model_outputs/personalization/figures/`.

In [ ]:
# Figure setup and shared visual conventions
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

FIGURE_DIR = OUTPUT_DIR / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

PROFILE_ORDER = ['NetC Distributed', 'NetC Hub-locked']
PROFILE_SHORT = {
    'NetC Distributed': 'Distributed',
    'NetC Hub-locked': 'Hub-locked',
}
PROFILE_COLORS = {
    'NetC Distributed': '#0072B2',
    'NetC Hub-locked': '#D55E00',
}
CONDITION_LABELS = {
    'control': 'Control',
    'default': 'Default',
    'reintervention': 'Reintervention',
}

FIGURE_RC = {
    'font.family': 'DejaVu Sans',
    'font.size': 9.5,
    'axes.titlesize': 10.5,
    'axes.labelsize': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': False,
    'legend.frameon': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
}


def save_thesis_figure(fig, stem):
    png_path = FIGURE_DIR / f'{stem}.png'
    pdf_path = FIGURE_DIR / f'{stem}.pdf'
    fig.savefig(png_path, dpi=400, bbox_inches='tight', facecolor='white')
    fig.savefig(pdf_path, bbox_inches='tight', facecolor='white')
    return png_path, pdf_path


print('Figure directory:', FIGURE_DIR)

In [ ]:
# Figure 1 — corrected network phenotype definition and condition balance
with plt.rc_context(FIGURE_RC):
    fig, axes = plt.subplots(
        1, 2, figsize=(11.4, 4.35),
        gridspec_kw={'width_ratios': [1.28, 1.0]},
    )

    # A: strict two-feature decision geometry, colored by the corrected NetC labels.
    ax = axes[0]
    strict_centers_z = (
        pd.DataFrame(X_net2, columns=NET2_FEATURES)
        .assign(cluster_net2=network_base['cluster_net2'].to_numpy())
        .groupby('cluster_net2')[NET2_FEATURES].mean().sort_index()
    )
    hhi_grid = np.linspace(0.10, 1.015, 320)
    rho_grid = np.linspace(0.42, 1.015, 320)
    gx, gy = np.meshgrid(hhi_grid, rho_grid)
    grid_z = strict_scaler.transform(np.column_stack([gx.ravel(), gy.ravel()]))
    distances = np.stack([
        np.sum((grid_z - strict_centers_z.loc[c].to_numpy()) ** 2, axis=1)
        for c in strict_centers_z.index
    ], axis=1)
    region = np.asarray(strict_centers_z.index)[np.argmin(distances, axis=1)].reshape(gx.shape)
    ax.contourf(
        gx, gy, region, levels=[-0.5, 0.5, 1.5],
        colors=['#DCEEF8', '#F8E4D8'], alpha=0.72,
    )

    point_mass = (network_base['HHI_net'] == 1.0) & (network_base['rho_self'] == 1.0)
    for profile in PROFILE_ORDER:
        mask = (network_base['profile_path_c'] == profile) & ~point_mass
        ax.scatter(
            network_base.loc[mask, 'HHI_net'],
            network_base.loc[mask, 'rho_self'],
            s=19, alpha=0.48, color=PROFILE_COLORS[profile],
            edgecolor='white', linewidth=0.25, rasterized=True,
            label=PROFILE_SHORT[profile],
        )

    mass_profile = network_base.loc[point_mass, 'profile_path_c'].iloc[0]
    ax.scatter(
        [1.0], [1.0], s=300, color=PROFILE_COLORS[mass_profile], alpha=0.72,
        edgecolor='black', linewidth=0.8, zorder=5,
    )
    ax.annotate(
        f'N={int(point_mass.sum())} at (1, 1)', xy=(1.0, 1.0), xytext=(0.76, 0.93),
        arrowprops={'arrowstyle': '-', 'color': '#555555', 'lw': 0.8},
        fontsize=8.5, color='#333333', ha='center',
    )

    for cluster_value, profile in [(0, PROFILE_ORDER[0]), (1, PROFILE_ORDER[1])]:
        center = strict_assignments.loc[
            strict_assignments['cluster_net2'] == cluster_value, NET2_FEATURES
        ].mean()
        ax.scatter(
            center['HHI_net'], center['rho_self'], marker='*', s=190,
            color=PROFILE_COLORS[profile], edgecolor='black', linewidth=0.9, zorder=6,
        )

    ax.set_xlim(0.10, 1.035)
    ax.set_ylim(0.42, 1.035)
    ax.set_xlabel(r'Network concentration ($HHI_{net}$)')
    ax.set_ylabel(r'Self-transition probability ($\rho_{self}$)')
    ax.set_title('A  Network phenotype geometry', loc='left', fontweight='bold')
    ax.text(
        0.02, 0.02,
        'Background: strict two-feature partition\n'
        f'NetC agreement: ARI={strict_vs_path_c_ari:.2f}',
        transform=ax.transAxes, va='bottom', fontsize=8.4,
        bbox={'boxstyle': 'round,pad=0.3', 'facecolor': 'white', 'alpha': 0.88, 'edgecolor': '#BBBBBB'},
    )
    ax.legend(loc='upper left', bbox_to_anchor=(0.0, 0.89), title='Corrected phenotype')

    # B: proportional composition by randomized study condition.
    ax = axes[1]
    balance_counts = pd.crosstab(
        network_base['study_condition'], network_base['profile_path_c']
    ).reindex(index=CONDITIONS, columns=PROFILE_ORDER, fill_value=0)
    balance_pct = balance_counts.div(balance_counts.sum(axis=1), axis=0) * 100
    bottom = np.zeros(len(CONDITIONS))
    x_positions = np.arange(len(CONDITIONS))
    for profile in PROFILE_ORDER:
        values = balance_pct[profile].to_numpy()
        ax.bar(
            x_positions, values, bottom=bottom, width=0.68,
            color=PROFILE_COLORS[profile], alpha=0.88,
            edgecolor='white', linewidth=0.8,
        )
        counts = balance_counts[profile].to_numpy()
        for x_pos, low, height, count in zip(x_positions, bottom, values, counts):
            ax.text(
                x_pos, low + height / 2, f'{count}\n({height:.0f}%)',
                ha='center', va='center', fontsize=8.5,
                color='white', fontweight='bold',
            )
        bottom += values

    ax.set_xticks(x_positions)
    ax.set_xticklabels([CONDITION_LABELS[c] for c in CONDITIONS])
    ax.set_ylim(0, 108)
    ax.set_ylabel('Phenotype composition within condition (%)')
    ax.set_title('B  Condition balance', loc='left', fontweight='bold')
    ax.text(
        0.02, 0.98,
        rf'$\chi^2$(2)={path_c_audit["chi2"]:.2f}, p={path_c_audit["p_value"]:.3f}'
        f'\nCramér\'s V={path_c_audit["cramers_v"]:.03f}; minimum cell={path_c_audit["min_observed"]}',
        transform=ax.transAxes, ha='left', va='top', fontsize=8.5,
        bbox={'boxstyle': 'round,pad=0.3', 'facecolor': 'white', 'alpha': 0.9, 'edgecolor': '#BBBBBB'},
    )
    handles = [Patch(facecolor=PROFILE_COLORS[p], label=PROFILE_SHORT[p]) for p in PROFILE_ORDER]
    ax.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, -0.24), ncol=2)

    fig.subplots_adjust(wspace=0.30, bottom=0.22)
    fig1_paths = save_thesis_figure(fig, 'fig1_network_phenotype_definition')
    plt.show()

print('Saved:', *[p.name for p in fig1_paths])

In [ ]:
# Figure 2 — raw distributions and corrected phenotype × condition patterns
outcome_figure_specs = [
    ('x_mean', 'Override rate', 'Primary', primary, (0.0, 1.02), False),
    ('commitment_score', 'Commitment score', 'Replication', replication, (0.0, 1.02), False),
    ('chi_behav', r'Behavioral susceptibility ($\chi_{behav}$)', 'Tertiary', tertiary, (0.85, 300), True),
]

with plt.rc_context(FIGURE_RC):
    fig, axes = plt.subplots(1, 3, figsize=(14.0, 4.75))
    jitter_rng = np.random.default_rng(RANDOM_STATE + 300)
    offsets = {'NetC Distributed': -0.17, 'NetC Hub-locked': 0.17}

    for panel_index, (ax, spec) in enumerate(zip(axes, outcome_figure_specs)):
        outcome, y_label, role, result, y_limits, log_scale = spec
        for profile in PROFILE_ORDER:
            positions = []
            medians = []
            for condition_index, condition in enumerate(CONDITIONS):
                values = analysis.loc[
                    (analysis['profile_path_c'] == profile) &
                    (analysis['study_condition'] == condition), outcome
                ].to_numpy(dtype=float)
                position = condition_index + offsets[profile]
                positions.append(position)
                medians.append(float(np.median(values)))
                jitter = jitter_rng.normal(0.0, 0.030, len(values))
                ax.scatter(
                    position + jitter, values, s=10, alpha=0.20,
                    color=PROFILE_COLORS[profile], edgecolor='none', rasterized=True,
                )
                box = ax.boxplot(
                    [values], positions=[position], widths=0.23,
                    patch_artist=True, showfliers=False, whis=(5, 95),
                    medianprops={'color': 'black', 'linewidth': 1.35},
                    boxprops={'edgecolor': PROFILE_COLORS[profile], 'linewidth': 1.0},
                    whiskerprops={'color': PROFILE_COLORS[profile], 'linewidth': 0.9},
                    capprops={'color': PROFILE_COLORS[profile], 'linewidth': 0.9},
                )
                box['boxes'][0].set_facecolor(PROFILE_COLORS[profile])
                box['boxes'][0].set_alpha(0.24)
            ax.plot(
                positions, medians, color=PROFILE_COLORS[profile],
                marker='o', markersize=5, linewidth=1.7, zorder=5,
            )

        if log_scale:
            ax.set_yscale('log')
            ax.set_yticks([1, 2, 5, 10, 20, 50, 100, 200])
            ax.yaxis.set_major_formatter(mticker.ScalarFormatter())
        ax.set_ylim(*y_limits)
        ax.set_xlim(-0.48, 2.48)
        ax.set_xticks(range(len(CONDITIONS)))
        ax.set_xticklabels([CONDITION_LABELS[c] for c in CONDITIONS], rotation=12, ha='right')
        ax.set_ylabel(y_label + (' (log display)' if log_scale else ''))
        ax.set_title(f'{chr(65 + panel_index)}  {role}: {outcome}', loc='left', fontweight='bold')
        ax.yaxis.grid(True, alpha=0.22, linestyle='--', linewidth=0.7)
        conclusion = 'small interaction' if result.interaction_p_family_adjusted < ALPHA else 'no detectable interaction'
        ax.text(
            0.03, 0.97,
            rf'ART $F$({int(result.interaction_df)}, {int(result.error_df)})={result.interaction_F:.2f}'
            f'\nHolm p={fmt_p(result.interaction_p_family_adjusted)}; '
            rf'$\eta_p^2$={result.partial_eta2:.3f}'
            f'\n{conclusion}',
            transform=ax.transAxes, ha='left', va='top', fontsize=8.4,
            bbox={'boxstyle': 'round,pad=0.3', 'facecolor': 'white', 'alpha': 0.9, 'edgecolor': '#BBBBBB'},
        )
        if log_scale:
            ax.text(
                0.98, 0.03, 'Log axis is visual only; ART uses raw ranks',
                transform=ax.transAxes, ha='right', va='bottom', fontsize=7.8,
                color='#555555', style='italic',
            )

    phenotype_handles = [
        Line2D([0], [0], color=PROFILE_COLORS[p], marker='o', lw=1.8,
               label=PROFILE_SHORT[p]) for p in PROFILE_ORDER
    ]
    fig.legend(
        handles=phenotype_handles, loc='lower center', ncol=2,
        bbox_to_anchor=(0.5, -0.02), title='Corrected NetC phenotype',
    )
    fig.text(
        0.01, -0.005,
        'Points = participants; boxes = IQR with 5th–95th percentile whiskers; connected markers = cell medians.',
        fontsize=8.2, color='#444444',
    )
    fig.subplots_adjust(wspace=0.32, bottom=0.25)
    fig2_paths = save_thesis_figure(fig, 'fig2_corrected_interaction_distributions')
    plt.show()

print('Saved:', *[p.name for p in fig2_paths])

In [ ]:
# Figure 3 — corrected interaction effect-size summary
forest_rows = interaction_results[
    interaction_results['analysis_id'].isin([
        'corrected_primary_path_c_x_mean',
        'corrected_replication_path_c_commitment_score',
        'corrected_tertiary_path_c_chi_behav',
        'corrected_sensitivity_strict_net2_x_mean',
    ])
].set_index('analysis_id').loc[[
    'corrected_primary_path_c_x_mean',
    'corrected_replication_path_c_commitment_score',
    'corrected_tertiary_path_c_chi_behav',
    'corrected_sensitivity_strict_net2_x_mean',
]].reset_index()

forest_labels = [
    r'Primary: $x_{mean}$',
    'Replication: commitment score',
    r'Tertiary: $\chi_{behav}$',
    r'Sensitivity: strict Net2 → $x_{mean}$',
]
forest_colors = ['#4C78A8', '#4C78A8', '#D55E00', '#777777']

with plt.rc_context(FIGURE_RC):
    fig, ax = plt.subplots(figsize=(9.2, 4.45))
    y_positions = np.arange(len(forest_rows))[::-1]
    estimates = forest_rows['partial_eta2'].to_numpy(float)
    low = forest_rows['partial_eta2_boot_ci_low'].to_numpy(float)
    high = forest_rows['partial_eta2_boot_ci_high'].to_numpy(float)

    ax.axvspan(0.0, 0.01, color='#F2F2F2', zorder=0)
    ax.axvspan(0.01, 0.06, color='#E8F1F7', alpha=0.65, zorder=0)
    ax.axvline(0.01, color='#888888', linestyle='--', linewidth=0.9)
    ax.axvline(0.06, color='#888888', linestyle=':', linewidth=0.9)
    ax.text(0.005, 3.36, 'below small', ha='center', va='top', fontsize=8, color='#666666')
    ax.text(0.035, 3.36, 'small', ha='center', va='top', fontsize=8, color='#52718A')
    ax.text(0.067, 3.36, 'medium threshold', ha='center', va='top', fontsize=8, color='#666666')

    for i, (y_pos, estimate, ci_low, ci_high, color) in enumerate(
        zip(y_positions, estimates, low, high, forest_colors)
    ):
        marker = 'o' if i < 3 else 'D'
        face = color if i < 3 else 'white'
        ax.errorbar(
            estimate, y_pos,
            xerr=[[estimate - ci_low], [ci_high - estimate]],
            fmt=marker, markersize=7.0, color=color, markerfacecolor=face,
            markeredgecolor=color, markeredgewidth=1.4,
            capsize=3.5, elinewidth=1.5, zorder=3,
        )
        row = forest_rows.iloc[i]
        p_text = (
            f'Holm p={fmt_p(row.interaction_p_family_adjusted)}'
            if i < 3 else f'raw p={fmt_p(row.interaction_p_raw)} (sensitivity)'
        )
        ax.text(0.078, y_pos, p_text, ha='left', va='center', fontsize=8.5)

    ax.axhline(0.5, color='#AAAAAA', linewidth=0.8, linestyle='--')
    ax.text(
        0.002, 0.52, 'three-test corrected family',
        transform=ax.get_yaxis_transform(), ha='left', va='bottom', fontsize=8, color='#555555'
    )
    ax.text(
        0.002, 0.48, 'non-discovery sensitivity',
        transform=ax.get_yaxis_transform(), ha='left', va='top', fontsize=8, color='#555555'
    )
    ax.set_yticks(y_positions)
    ax.set_yticklabels(forest_labels)
    ax.set_xlim(0, 0.112)
    ax.set_ylim(-0.15, 3.48)
    ax.set_xlabel(r'Interaction partial $\eta_p^2$ (stratified bootstrap 95% CI)')
    ax.set_title('Corrected network-phenotype interaction effects', loc='left', fontweight='bold', pad=10)
    ax.xaxis.grid(True, alpha=0.22, linestyle='--', linewidth=0.7)
    ax.text(
        0.0, -0.25,
        'Legacy profile_p1/profile_k3 effects are not shown because moderator–outcome feature leakage makes them non-confirmatory.\n'
        'Bootstrap intervals for η² are bounded at zero; inferential conclusions use the ART interaction p-values and the frozen Holm family.',
        transform=ax.transAxes, fontsize=8.1, color='#444444', va='top',
    )
    fig.subplots_adjust(left=0.29, right=0.98, bottom=0.29, top=0.86)
    fig3_paths = save_thesis_figure(fig, 'fig3_corrected_effect_sizes')
    plt.show()

print('Saved:', *[p.name for p in fig3_paths])

In [ ]:
# Document figure selection, interpretation, and deliberate exclusions
figure_selection_text = f"""# DS5 Step 3 — Figure Selection and Interpretation

**Produced by:** `notebooks/200_ds5_personalization_exploration.ipynb`  
**Evidence tier visualized:** corrected NetC / strict outcome-free network moderator only  
**Formats:** 400-dpi PNG and vector PDF

## Chosen figures

### Figure 1 — Corrected network phenotype definition and condition balance

Files: `figures/fig1_network_phenotype_definition.png` and `.pdf`

**Why it is included.** The moderator is scientifically meaningful only if readers can see what separates its two network phenotypes and why it is eligible for cross-condition interaction testing. This combines those two prerequisite ideas without adding a standalone methods-only diagnostic figure.

**What it shows.** Panel A plots `HHI_net` against `rho_self`, marks network centroids, and overlays the strict two-feature sensitivity partition. The pre-existing four-feature NetC assignments agree exactly with that strict partition in the N=414 analysis sample (ARI={strict_vs_path_c_ari:.2f}). The large point at (1,1) is explicitly sized/annotated as an observed point mass rather than hidden by overplotting. Panel B shows near-constant phenotype composition across conditions (chi-square p={path_c_audit['p_value']:.3f}, Cramer's V={path_c_audit['cramers_v']:.3f}; minimum cell N={path_c_audit['min_observed']}). This supports condition independence, not pre-treatment status or causality.

### Figure 2 — Corrected outcome distributions and interaction patterns

Files: `figures/fig2_corrected_interaction_distributions.png` and `.pdf`

**Why it is included.** A table of six cell summaries per outcome does not reveal skew, overlap, outliers, or the scale of between-participant variation. Showing all participants, robust summaries, and cell medians makes both primary nulls visible rather than presenting only the significant tertiary outcome.

**What it shows.** The primary `x_mean` and replication `commitment_score` panels show substantial distributional overlap and no detectable phenotype-by-condition interaction (Holm p={primary.interaction_p_family_adjusted:.3f} and {replication.interaction_p_family_adjusted:.3f}). The tertiary `chi_behav` panel shows a small interaction (Holm p={tertiary.interaction_p_family_adjusted:.3f}, partial eta-squared={tertiary.partial_eta2:.3f}) amid wide, strongly right-skewed distributions. A logarithmic display axis prevents a few extreme values from visually dominating; the ART inference itself remains based on raw ranks. Placing all three outcomes on equal-sized panels prevents the tertiary result from receiving disproportionate visual emphasis.

### Figure 3 — Corrected interaction effect-size summary

Files: `figures/fig3_corrected_effect_sizes.png` and `.pdf`

**Why it is included.** This is the most compact inferential summary and makes magnitude—not only significance—visible. It also clearly separates the frozen three-test corrected family from the non-discovery strict sensitivity.

**What it shows.** The primary and replication effects are below the conventional small threshold, whereas `chi_behav` is small. The strict two-feature sensitivity reproduces the primary point estimate exactly because its assignments equal NetC on all 414 participants. Confidence intervals are shown for uncertainty, while the adjacent Holm/raw p-values carry the interaction-test conclusion. Legacy leakage-affected estimates are explicitly omitted from this claim-bearing forest and remain available in the provenance table.

## Candidate figures deliberately not included

- **A standalone `chi_behav` hero figure:** rejected because it would visually elevate a tertiary, small effect above the null primary and replication results.
- **Legacy `profile_p1` / `profile_k3` interaction figures:** rejected because feature overlap makes those analyses non-confirmatory. Plotting them beside corrected results would create false evidential equivalence.
- **A NetC-versus-strict alluvial/confusion plot:** rejected because ARI=1.00 makes the mapping perfectly diagonal and scientifically trivial; the agreement is stated directly in Figure 1 and Figure 3.
- **A separate k-selection or stability dashboard:** rejected because the selected solution is unambiguous and near-perfectly stable (Jaccard={strict_jaccard_mean:.3f}). The diagnostic table remains the appropriate detailed record.
- **Radar charts:** rejected because only two strict defining features are relevant, making a radar geometry decorative and potentially distorting relative separation.
- **Condition-by-phenotype mean bars without raw data:** rejected because bars would hide the severe skew and individual heterogeneity, especially for `chi_behav`.
- **Recommendation heatmaps or “best condition” maps:** rejected because DS5 does not validate phenotype-guided assignment and such a graphic would imply an unsupported prescriptive rule.
- **A separate figure for the strict sensitivity outcome:** rejected because its assignments and `x_mean` interaction are identical to the corrected primary; Figure 3 records this robustness result without duplicating Figure 2.

## Interpretation boundary

These figures communicate observed group-level effect heterogeneity under an outcome-free network moderator. They do not establish that the moderator is pre-treatment, that phenotype causes differential response, or that assigning a condition by phenotype improves individual outcomes.
"""

(OUTPUT_DIR / 'figure_selection_and_interpretation.md').write_text(figure_selection_text)
print('Saved figure_selection_and_interpretation.md')

In [ ]:
# 15. Export primary tables and companion documentation
interaction_results.to_csv(
    OUTPUT_DIR / 'phenotype_condition_interaction_results.csv',
    index=False, float_format='%.10g',
)
cell_summary.to_csv(
    OUTPUT_DIR / 'phenotype_condition_cell_summary.csv',
    index=False, float_format='%.10g',
)

result_table_md = markdown_table(
    interaction_results,
    columns=[
        'result_tier', 'analysis_role', 'moderator', 'outcome',
        'interaction_F', 'interaction_df', 'error_df',
        'interaction_p_raw', 'interaction_p_family_adjusted',
        'partial_eta2', 'partial_eta2_boot_ci_low',
        'partial_eta2_boot_ci_high', 'feature_overlap_flag', 'claim_eligible',
    ],
)

interaction_doc = f"""# phenotype_condition_interaction_results.csv

**Produced by:** `notebooks/200_ds5_personalization_exploration.ipynb`  
**Rows:** 8 — three corrected family tests, one corrected sensitivity, four legacy provenance tests

## Corrected multiplicity definition

The NetC `x_mean`, `commitment_score`, and `chi_behav` interactions form one prospectively defined three-test Holm family. The strict two-feature `x_mean` analysis is a non-discovery robustness check and is not added to that family. Legacy tests retain their original four-test Holm values for provenance only.

## Results

{result_table_md}

## Method

Cellwise Shapiro-Wilk and median-centered Levene tests select raw OLS only if all assumptions pass; otherwise the aligned rank transform interaction is used. Partial eta-squared receives a {N_BOOTSTRAPS}-iteration stratified percentile bootstrap CI. Both OLS and ART estimates remain in the CSV.

## Claim boundary

Only rows marked `claim_eligible=True` belong to the corrected evidence tier. Sensitivity and legacy rows cannot independently establish a discovery. Legacy `profile_p1`/`profile_k3` rows are explicitly non-confirmatory because of moderator-outcome leakage.
"""

cell_doc = f"""# phenotype_condition_cell_summary.csv

**Produced by:** `notebooks/200_ds5_personalization_exploration.ipynb`  
**Rows:** {len(cell_summary)} phenotype x condition x outcome descriptive cells

The original 168 Framework B rows are retained. Thirty-six corrected rows add NetC and strict Net2 summaries across all three outcomes. Every row contains N, mean, median, SD, quartiles, and {N_BOOTSTRAPS}-iteration within-cell bootstrap CIs.

Framework B is descriptive. Path A, Path F, Path E, and GaT strategy remain condition-shaped or condition-specific and cannot support trait-moderation claims. Legacy Phase 1/GMM rows carry explicit feature-leakage warnings.
"""

(OUTPUT_DIR / 'phenotype_condition_interaction_results.md').write_text(interaction_doc)
(OUTPUT_DIR / 'phenotype_condition_cell_summary.md').write_text(cell_doc)
print('Saved corrected interaction and Framework B tables/documentation.')

In [ ]:
# 16. Document strict phenotype outputs
strict_assignment_doc = f"""# strict_net2_assignments.csv

**Produced by:** `notebooks/200_ds5_personalization_exploration.ipynb`  
**Role:** Pre-specified corrected Step 3 sensitivity moderator  
**Population:** N={EXPECTED_N}

## Construction

- Inputs exactly: `HHI_net`, `rho_self`
- No behavioral variables loaded during construction or naming
- Complete-case N=414; no imputation
- StandardScaler fitted on N=414
- K-means++ with n_init=10 and random_state=42
- Candidate k=2..6; original Phase 1 rule: silhouette >0.20, minimum cluster >=40, then maximum silhouette
- Selected k={SELECTED_NET2_K}
- Labels ordered by HHI_net/rho_self centroids only

## Gates

- Bootstrap co-membership Jaccard={strict_jaccard_mean:.4f} ± {strict_jaccard_std:.4f}
- Condition chi2({strict_audit['df']})={strict_audit['chi2']:.4f}, p={strict_audit['p_value']:.4f}, Cramer's V={strict_audit['cramers_v']:.4f}
- Minimum phenotype x condition cell={strict_audit['min_observed']}
- All pre-outcome gates passed: {strict_gate_pass}
- ARI versus pre-existing NetC labels on N=414: {strict_vs_path_c_ari:.4f}

The network metrics were measured during the study period, so this is outcome-nonoverlapping but not established as a pre-treatment phenotype.
"""

strict_diagnostic_doc = f"""# strict_net2_cluster_diagnostics.csv

Diagnostics for k=2..6 using exactly standardized `HHI_net` and `rho_self`. `selected=True` identifies the solution chosen under the frozen Phase 1 rule. Stability and condition-independence fields are populated only for the selected solution because these gates are applied after k selection and before outcome loading.

Selected k={SELECTED_NET2_K}; silhouette={strict_diagnostics.loc[strict_diagnostics.selected, 'silhouette'].iloc[0]:.4f}; Jaccard={strict_jaccard_mean:.4f}; condition p={strict_audit['p_value']:.4f}; minimum condition cell={strict_audit['min_observed']}.
"""

(OUTPUT_DIR / 'strict_net2_assignments.md').write_text(strict_assignment_doc)
(OUTPUT_DIR / 'strict_net2_cluster_diagnostics.md').write_text(strict_diagnostic_doc)
print('Saved strict phenotype companion documentation.')

In [ ]:
# 17. Write corrected model_run_summary.md
corrected_cells = cell_summary[
    (cell_summary['moderator'] == 'profile_path_c')
][['outcome', 'phenotype_level', 'study_condition', 'n', 'mean', 'median']]

strict_cells = cell_summary[
    (cell_summary['moderator'] == 'profile_net2') &
    (cell_summary['outcome'] == 'x_mean')
][['phenotype_level', 'study_condition', 'n', 'mean', 'median']]

legacy_table = interaction_results[
    interaction_results['result_tier'] == 'legacy_provenance_nonconfirmatory'
][[
    'analysis_role', 'moderator', 'outcome', 'interaction_F',
    'interaction_p_raw', 'interaction_p_family_adjusted',
    'partial_eta2', 'feature_overlap_reason',
]]

all_behavior_level_null = bool(
    primary.interaction_p_family_adjusted >= ALPHA and
    replication.interaction_p_family_adjusted >= ALPHA and
    sensitivity.interaction_p_raw >= ALPHA
)

summary_text = f"""# DS5 Step 3 — Corrected Personalization Model Run Summary

**Run date:** {datetime.date.today().isoformat()}  
**Notebook:** `notebooks/200_ds5_personalization_exploration.ipynb`  
**Analysis population:** N={EXPECTED_N}

---

## Corrected design

- Primary moderator: pre-existing NetC `cluster_net_only`, semantically named from network centroids only
- Corrected Holm family: NetC interactions on `x_mean`, `commitment_score`, and `chi_behav` (3 tests)
- Sensitivity: strict `HHI_net` + `rho_self` phenotype on `x_mean`; non-discovery robustness check outside the family
- Legacy `profile_p1` and `profile_k3` interactions retained as non-confirmatory provenance

## Moderator gates

### Pre-existing NetC

- Original fitting population: N=463 NeT-eligible participants
- Inputs: `HHI_net`, `rho_self`, `H_trans`, `H_app`
- Full-sample Jaccard={float(path_c_full_stability.jaccard_mean):.4f} ± {float(path_c_full_stability.jaccard_std):.4f}
- Step 3 overlap cells: distributed 42/46/36; hub-locked 95/96/99
- Condition chi2({path_c_audit['df']})={path_c_audit['chi2']:.4f}, p={path_c_audit['p_value']:.4f}, Cramer's V={path_c_audit['cramers_v']:.4f}

### Strict two-feature sensitivity

- Inputs exactly `HHI_net`, `rho_self`; behavioral outcomes unavailable during construction/naming
- Selected k={SELECTED_NET2_K}; silhouette={strict_diagnostics.loc[strict_diagnostics.selected, 'silhouette'].iloc[0]:.4f}
- Bootstrap Jaccard={strict_jaccard_mean:.4f} ± {strict_jaccard_std:.4f}
- Cells: distributed 42/46/36; hub-locked 95/96/99
- Condition chi2({strict_audit['df']})={strict_audit['chi2']:.4f}, p={strict_audit['p_value']:.4f}, Cramer's V={strict_audit['cramers_v']:.4f}
- ARI versus NetC on N=414={strict_vs_path_c_ari:.4f}; the assignments are identical on the analysis population

## Assumption and robustness methods

All corrected analyses failed the raw-cell normality gate and therefore use the aligned rank transform interaction. Raw OLS results remain exported. Partial eta-squared uses {N_BOOTSTRAPS} stratified bootstrap iterations.

## Corrected results

### Primary — NetC x condition -> x_mean

- ART F({int(primary.interaction_df)}, {int(primary.error_df)})={primary.interaction_F:.4f}
- Raw p={fmt_p(primary.interaction_p_raw)}; corrected-family Holm p={fmt_p(primary.interaction_p_family_adjusted)}
- Partial eta-squared={primary.partial_eta2:.4f} ({primary.partial_eta2_label}); bootstrap 95% CI [{primary.partial_eta2_boot_ci_low:.4f}, {primary.partial_eta2_boot_ci_high:.4f}]
- **Result: no detectable network-phenotype moderation of override rate.**

### Replication — NetC x condition -> commitment_score

- ART F({int(replication.interaction_df)}, {int(replication.error_df)})={replication.interaction_F:.4f}
- Raw p={fmt_p(replication.interaction_p_raw)}; corrected-family Holm p={fmt_p(replication.interaction_p_family_adjusted)}
- Partial eta-squared={replication.partial_eta2:.4f} ({replication.partial_eta2_label}); bootstrap 95% CI [{replication.partial_eta2_boot_ci_low:.4f}, {replication.partial_eta2_boot_ci_high:.4f}]
- **Result: no detectable interaction; the independently computed proxy agrees with the primary null.**

### Tertiary — NetC x condition -> chi_behav

- ART F({int(tertiary.interaction_df)}, {int(tertiary.error_df)})={tertiary.interaction_F:.4f}
- Raw p={fmt_p(tertiary.interaction_p_raw)}; corrected-family Holm p={fmt_p(tertiary.interaction_p_family_adjusted)}
- Partial eta-squared={tertiary.partial_eta2:.4f} ({tertiary.partial_eta2_label}); bootstrap 95% CI [{tertiary.partial_eta2_boot_ci_low:.4f}, {tertiary.partial_eta2_boot_ci_high:.4f}]
- Pairwise interaction follow-ups: {corrected_chi_pair_text}
- **Result: small condition-dependent association in ranked susceptibility distributions.** Unlike the legacy result, the NetC moderator has no defining-feature overlap with chi_behav.

### Strict HHI_net + rho_self sensitivity -> x_mean

- ART F({int(sensitivity.interaction_df)}, {int(sensitivity.error_df)})={sensitivity.interaction_F:.4f}
- Unadjusted sensitivity p={fmt_p(sensitivity.interaction_p_raw)}; no separate discovery adjustment/claim
- Partial eta-squared={sensitivity.partial_eta2:.4f} ({sensitivity.partial_eta2_label}); bootstrap 95% CI [{sensitivity.partial_eta2_boot_ci_low:.4f}, {sensitivity.partial_eta2_boot_ci_high:.4f}]
- **Result: identical to the corrected primary because the strict and NetC labels coincide for all 414 participants.**

## Do the corrected analyses agree?

- NetC primary and strict two-feature sensitivity agree exactly on the `x_mean` null.
- `commitment_score` agrees with the behavioral-level null.
- `chi_behav` does not share that null conclusion: it retains a small corrected-family interaction.
- Therefore, all corrected analyses agree that there is no phenotype moderation of the behavioral level, but not all outcomes are null; susceptibility may vary by condition and network phenotype.
- **Independence caveat:** the corrected moderator NetC and the legacy moderator `profile_p1` assign 413 of 414 participants identically (ARI=0.9901; the single disagreement is one control participant). The corrected `chi_behav` result is therefore a clean-provenance re-run of the same partition, not independent corroboration of the legacy result — the default-vs-reintervention post-hoc F=8.0149 is numerically identical in both tables for exactly this reason.

## Corrected NetC descriptive cells

{markdown_table(corrected_cells, digits=4)}

## Strict two-feature x_mean cells

{markdown_table(strict_cells, digits=4)}

## Publication figures

### Figure 1 — Corrected network phenotype definition and condition balance

- Files: `figures/fig1_network_phenotype_definition.png` and `.pdf`
- **Why included:** establishes what the eligible moderator represents and shows the condition-balance prerequisite in one compact figure.
- **Scientific reading:** the strict `HHI_net` + `rho_self` geometry reproduces NetC exactly on N=414 (ARI=1.00); phenotype composition is similar across conditions (p=0.568, minimum cell N=36). This establishes outcome non-overlap and condition independence, not pre-treatment causality.

### Figure 2 — Corrected outcome distributions and interaction patterns

- Files: `figures/fig2_corrected_interaction_distributions.png` and `.pdf`
- **Why included:** exposes participant-level skew and overlap while giving the null primary and replication the same visual prominence as the tertiary result.
- **Scientific reading:** `x_mean` and `commitment_score` show no detectable interaction; `chi_behav` shows a small interaction amid broad right-skewed distributions. The `chi_behav` axis is logarithmic for display only; ART inference uses raw ranks.

### Figure 3 — Corrected interaction effect sizes

- Files: `figures/fig3_corrected_effect_sizes.png` and `.pdf`
- **Why included:** compares magnitude and uncertainty directly and separates the frozen three-test corrected family from the non-discovery sensitivity.
- **Scientific reading:** primary and replication effects are below the conventional small threshold; `chi_behav` is small; strict Net2 reproduces the primary estimate exactly. Legacy leakage-affected effects are deliberately absent from this claim-bearing forest.

### Deliberate figure exclusions

Not included: a standalone `chi_behav` hero plot, legacy `profile_p1`/`profile_k3` plots, a trivial NetC-versus-strict alluvial plot, a separate k/stability dashboard, radar charts, raw-data-free mean bars, recommendation heatmaps, or a duplicate strict-sensitivity outcome panel. These candidates would respectively exaggerate a tertiary result, create false evidential equivalence, repeat deterministic information, add methods clutter, distort a two-feature structure, hide skew, imply unsupported prescription, or duplicate the primary figure. Full rationale is in `figure_selection_and_interpretation.md`.

## Legacy provenance — explicitly non-confirmatory

{markdown_table(legacy_table, digits=4)}

The legacy primary used `profile_p1`, which was constructed from `sig_hat`, `H_emp`, `x_mean`, `HHI_net`, and `rho_self` and then named by ascending `x_mean`. `chi_behav` is deterministic in `sig_hat` and `x_mean`; `commitment_score` correlates -0.9797 with `x_mean`; `profile_k3` used the same five inputs. These results remain reproducible historical computations but cannot support personalization evidence. Note also that `profile_p1` and the corrected NetC moderator coincide on 413/414 participants (ARI=0.9901), so swapping moderators removes the leaky *provenance* but does not test the partition against an independent one.

## Strongest defensible Step 3 claim

In this DS5 sample, the pre-existing outcome-free NetC network phenotype does **not** identify groups with detectably different condition effects on override rate or the commitment proxy. It does show a small condition-dependent association with ranked behavioral susceptibility after three-test Holm correction. This is group-level, exploratory effect heterogeneity—not an individual prescription—and the network phenotype was measured during the study period rather than established pre-treatment.

## Strongest claim DS5 still cannot support

DS5 cannot support a validated per-user recommendation, nor the causal claim that phenotype-guided condition assignment would improve outcomes. Each participant was observed under one condition, the moderator is a study-period observational phenotype, no phenotype-stratified policy RCT was run, and the failed ABM gate prevents credible simulated individual counterfactuals.

## Output files

- `phenotype_condition_interaction_results.csv` + `.md`
- `phenotype_condition_cell_summary.csv` + `.md`
- `strict_net2_assignments.csv` + `.md`
- `strict_net2_cluster_diagnostics.csv` + `.md`
- `model_run_summary.md`
- `figure_selection_and_interpretation.md`
- `figures/fig1_network_phenotype_definition.png` + `.pdf`
- `figures/fig2_corrected_interaction_distributions.png` + `.pdf`
- `figures/fig3_corrected_effect_sizes.png` + `.pdf`
"""

(OUTPUT_DIR / 'model_run_summary.md').write_text(summary_text)
print('Saved corrected model_run_summary.md')

In [ ]:
# 18. End-to-end output validation
expected_outputs = [
    OUTPUT_DIR / 'phenotype_condition_interaction_results.csv',
    OUTPUT_DIR / 'phenotype_condition_interaction_results.md',
    OUTPUT_DIR / 'phenotype_condition_cell_summary.csv',
    OUTPUT_DIR / 'phenotype_condition_cell_summary.md',
    OUTPUT_DIR / 'strict_net2_assignments.csv',
    OUTPUT_DIR / 'strict_net2_assignments.md',
    OUTPUT_DIR / 'strict_net2_cluster_diagnostics.csv',
    OUTPUT_DIR / 'strict_net2_cluster_diagnostics.md',
    OUTPUT_DIR / 'model_run_summary.md',
    OUTPUT_DIR / 'figure_selection_and_interpretation.md',
    FIGURE_DIR / 'fig1_network_phenotype_definition.png',
    FIGURE_DIR / 'fig1_network_phenotype_definition.pdf',
    FIGURE_DIR / 'fig2_corrected_interaction_distributions.png',
    FIGURE_DIR / 'fig2_corrected_interaction_distributions.pdf',
    FIGURE_DIR / 'fig3_corrected_effect_sizes.png',
    FIGURE_DIR / 'fig3_corrected_effect_sizes.pdf',
]
for path in expected_outputs:
    assert path.exists() and path.stat().st_size > 0, f'Missing/empty: {path}'

check_results = pd.read_csv(OUTPUT_DIR / 'phenotype_condition_interaction_results.csv')
check_cells = pd.read_csv(OUTPUT_DIR / 'phenotype_condition_cell_summary.csv')
check_strict = pd.read_csv(OUTPUT_DIR / 'strict_net2_assignments.csv')
check_diagnostics = pd.read_csv(OUTPUT_DIR / 'strict_net2_cluster_diagnostics.csv')

assert len(check_results) == 8
assert len(check_results[check_results.result_tier == 'corrected_inferential_family']) == 3
assert len(check_results[check_results.result_tier == 'legacy_provenance_nonconfirmatory']) == 4
assert len(check_cells) == 204
assert len(check_strict) == EXPECTED_N and check_strict.participant_id_canonical.is_unique
assert set(check_strict.profile_net2) == {'Net2 Distributed', 'Net2 Hub-locked'}
assert len(check_diagnostics) == 5
assert check_diagnostics.selected.sum() == 1
assert bool(check_diagnostics.loc[check_diagnostics.selected, 'all_preoutcome_gates_pass'].iloc[0])
assert np.isclose(strict_vs_path_c_ari, 1.0)
assert np.isclose(primary.interaction_F, sensitivity.interaction_F)
assert np.isclose(primary.interaction_p_raw, sensitivity.interaction_p_raw)
for figure_path in expected_outputs[-6:]:
    assert figure_path.stat().st_size > 10_000, f'Figure unexpectedly small: {figure_path}'
assert 'Candidate figures deliberately not included' in (OUTPUT_DIR / 'figure_selection_and_interpretation.md').read_text()

legacy_expected_ids = {
    'legacy_primary_profile_p1_x_mean',
    'legacy_replication_profile_p1_commitment_score',
    'legacy_tertiary_profile_p1_chi_behav',
    'legacy_sensitivity_profile_k3_x_mean',
}
assert set(check_results.loc[
    check_results.result_tier == 'legacy_provenance_nonconfirmatory', 'analysis_id'
]) == legacy_expected_ids

print('END-TO-END CORRECTED STEP 3 VALIDATION PASSED')
for path in expected_outputs:
    print(f'  {path.relative_to(ROOT)} ({path.stat().st_size:,} bytes)')

display(Markdown(
    f"""**Corrected primary:** ART F({int(primary.interaction_df)}, {int(primary.error_df)})={primary.interaction_F:.3f}, Holm p={fmt_p(primary.interaction_p_family_adjusted)} — null.  
**Replication:** ART F({int(replication.interaction_df)}, {int(replication.error_df)})={replication.interaction_F:.3f}, Holm p={fmt_p(replication.interaction_p_family_adjusted)} — null.  
**Tertiary chi_behav:** ART F({int(tertiary.interaction_df)}, {int(tertiary.error_df)})={tertiary.interaction_F:.3f}, Holm p={fmt_p(tertiary.interaction_p_family_adjusted)} — small interaction.  
**Strict Net2 sensitivity:** ART F({int(sensitivity.interaction_df)}, {int(sensitivity.error_df)})={sensitivity.interaction_F:.3f}, p={fmt_p(sensitivity.interaction_p_raw)} — identical primary null."""
))